# Monocular Visual Odometry — KITTI Evaluation

**Pipeline:** Harris → ORB → Hamming BFMatcher + ratio test → RANSAC Essential Matrix → recoverPose → Triangulation → Trajectory

**Before running:**
1. Add your KITTI dataset via *+ Add Data* in the sidebar
2. Set `GITHUB_REPO` in Cell 1 to your actual repo URL
3. Enable *Internet* in notebook settings (right panel)
4. Run all cells top to bottom

In [32]:
# ============================================================
#  EDIT THESE — everything else is automatic
# ============================================================
GITHUB_REPO = "https://github.com/infi9itea/h-orb-vo.git"

# Kaggle mounts datasets at /kaggle/input/<dataset-name>/
# Cell 2 will auto-detect the right path — or set it manually here.
KITTI_ROOT  = "/kaggle/input/datasets/hocop1/kitti-odometry"   # e.g. "/kaggle/input/kitti-odometry"

# Sequences to run. Start with ["00"] to verify, then expand.
SEQUENCES   = ["00", "01", "02", "03", "04", "05", "06", "07", "08", "09", "10"]

# Set to a small number (e.g. 200) while debugging, None for full run
MAX_FRAMES  = None

OUTPUT_DIR  = "/kaggle/working/results"
REPO_DIR    = "/kaggle/working/mono-vo"

## Cell 2 — Auto-detect KITTI dataset path

In [33]:
import os
from pathlib import Path

print('=== /kaggle/input contents ===')
for name in sorted(os.listdir('/kaggle/input')):
    full = f'/kaggle/input/{name}'
    children = sorted(os.listdir(full))[:6]
    print(f'  {full}/')
    for c in children:
        print(f'    {c}')

def find_kitti_root(base='/kaggle/input'):
    for root, dirs, files in os.walk(base):
        dirs.sort()
        if 'sequences' in dirs:
            return root
        if root.count(os.sep) - base.count(os.sep) > 3:
            dirs.clear()
    return None

if KITTI_ROOT is None:
    detected = find_kitti_root()
    if detected:
        KITTI_ROOT = detected
        print(f'\nAuto-detected KITTI root: {KITTI_ROOT}')
    else:
        print('\nCould not auto-detect. Set KITTI_ROOT manually in Cell 1.')
else:
    print(f'\nUsing KITTI_ROOT = {KITTI_ROOT}')

# Validate it looks right
if KITTI_ROOT:
    seq_dir = Path(KITTI_ROOT) / 'sequences'
    if seq_dir.exists():
        seqs_found = sorted(os.listdir(seq_dir))
        print(f'Sequences found: {seqs_found}')
    else:
        print(f'WARNING: no sequences/ folder inside {KITTI_ROOT}')

=== /kaggle/input contents ===
  /kaggle/input/datasets/
    hocop1

Using KITTI_ROOT = /kaggle/input/datasets/hocop1/kitti-odometry
Sequences found: ['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21']


## Cell 3 — Install dependencies

In [34]:
import subprocess, sys

def run(cmd, check=True):
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-500:])
        if check:
            raise RuntimeError(f'Command failed: {cmd}')
    return result

# Kaggle already ships numpy + opencv; we just need contrib for ORB + scipy
run('pip install opencv-contrib-python scipy --quiet')
print('Dependencies ready.')

Dependencies ready.


## Cell 4 — Pull code from GitHub

In [35]:
import os

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest...')
    run(f'git -C {REPO_DIR} pull')
else:
    print('Cloning repo...')
    run(f'git clone {GITHUB_REPO} {REPO_DIR}')

result = run(f'git -C {REPO_DIR} log --oneline -5', check=False)
print('Latest commits:')
print(result.stdout)

sys.path.insert(0, f'{REPO_DIR}/src')
print('Code ready.')

Repo exists — pulling latest...
Already up to date.

cd5f135 Delete README.md content
7e1d259 Implement Monocular Visual Odometry Pipeline with Metrics and Visualization
bd5ae79 Initial commit

Latest commits:
cd5f135 Delete README.md content
7e1d259 Implement Monocular Visual Odometry Pipeline with Metrics and Visualization
bd5ae79 Initial commit

Code ready.


## Cell 5 — Run test suite

In [36]:
result = run(f'python {REPO_DIR}/tests/test_pipeline.py', check=False)
if '0 failed' in result.stdout:
    print('All tests passed — pipeline is healthy.')
else:
    print('WARNING: some tests failed. Check output before continuing.')


Running mono_vo test suite
----------------------------------------
test_harris_detection ... OK  (500 keypoints)
test_orb_descriptor ... OK  (156 keypoints with descriptors)
test_hamming_matcher ... OK  (22 matches after ratio test)
test_essential_matrix ... OK  (rot_err=0.47 deg, inliers=93)
test_triangulation ... OK  (3-D error=0.0000 m)
test_full_pipeline_synthetic ... OK  (final pos=[[ 0.9934462  -0.03173552  0.10980666]])
test_metrics_ate ... OK
test_metrics_rpe ... OK

----------------------------------------
  8 passed, 0 failed

All tests passed — pipeline is healthy.


## Cell 6 — Verify dataset structure

In [37]:
from pathlib import Path

root = Path(KITTI_ROOT)
ok_seqs, skip_seqs = [], []

# KITTI can sometimes use image_0 (grayscale), image_2 (color), etc.
# We will try to find the first available image directory if image_0 is missing.
IMAGE_DIR_NAME = 'image_0'

for seq in SEQUENCES:
    seq = seq.zfill(2)
    seq_dir = root / 'sequences' / seq
    
    # Try to find a valid image directory if image_0 doesn't work
    img_dir = seq_dir / IMAGE_DIR_NAME
    if not img_dir.exists():
        possible_dirs = list(seq_dir.glob('image_*'))
        if possible_dirs:
            img_dir = possible_dirs[0]
            IMAGE_DIR_NAME = img_dir.name

    calib   = seq_dir / 'calib.txt'
    pose    = root / 'poses' / f'{seq}.txt'

    n_imgs    = len(list(img_dir.glob('*.png'))) if img_dir.exists() else 0
    has_calib = calib.exists()
    has_pose  = pose.exists()
    ok        = n_imgs > 0 and has_calib

    status = 'OK    ' if ok else 'MISSING'
    (ok_seqs if ok else skip_seqs).append(seq)
    print(f'  seq {seq}: images={n_imgs:4d} (in {IMAGE_DIR_NAME})  calib={"YES" if has_calib else "NO":3s}'
          f'  poses={"YES" if has_pose else "NO":3s}  [{status}]')

# Only run sequences that exist
SEQUENCES = ok_seqs
if skip_seqs:
    print(f'\nSkipping missing sequences: {skip_seqs}')
print(f'Will run: {SEQUENCES}')

  seq 00: images=4541 (in image_3)  calib=YES  poses=YES  [OK    ]
  seq 01: images=1101 (in image_3)  calib=YES  poses=YES  [OK    ]
  seq 02: images=4661 (in image_3)  calib=YES  poses=YES  [OK    ]
  seq 03: images= 801 (in image_3)  calib=YES  poses=YES  [OK    ]
  seq 04: images= 271 (in image_3)  calib=YES  poses=YES  [OK    ]
  seq 05: images=2761 (in image_3)  calib=YES  poses=YES  [OK    ]
  seq 06: images=1101 (in image_3)  calib=YES  poses=YES  [OK    ]
  seq 07: images=1101 (in image_3)  calib=YES  poses=YES  [OK    ]
  seq 08: images=4071 (in image_3)  calib=YES  poses=YES  [OK    ]
  seq 09: images=1591 (in image_3)  calib=YES  poses=YES  [OK    ]
  seq 10: images=1201 (in image_3)  calib=YES  poses=YES  [OK    ]
Will run: ['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10']


## Cell 7 — Run VO pipeline on all sequences

In [ ]:
import time
import numpy as np
import os
from pathlib import Path

from vo_pipeline  import MonocularVOPipeline
from kitti_loader import KITTISequence, build_pipeline_for_kitti
from metrics      import compute_ate, compute_rpe, compute_drift, print_metrics
from visualize    import plot_trajectory_2d, plot_ate_over_time, plot_diagnostics

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
all_metrics = {}

for seq_id in SEQUENCES:
    seq_id = seq_id.zfill(2)
    print(f"\n{'='*60}")
    print(f'  KITTI Sequence {seq_id}')
    print(f"{'='*60}")
    try:
        # Determine which camera/folder to use by checking existence
        # KITTI usually has image_0, image_1, image_2, image_3
        camera_id = 0
        for cid in [0, 2, 1, 3]:
            test_path = Path(KITTI_ROOT) / 'sequences' / seq_id / f'image_{cid}'
            if test_path.exists() and any(test_path.iterdir()):
                camera_id = cid
                break
        
        print(f'  Using camera_id={camera_id}')
        seq      = KITTISequence(KITTI_ROOT, seq_id, camera_id=camera_id)
        pipeline = build_pipeline_for_kitti(seq)
        n_frames = min(len(seq), MAX_FRAMES or len(seq))
        t_start  = time.time()

        for idx in range(n_frames):
            frame = seq.get_frame(idx)
            if frame is None:
                continue
            pipeline.process_frame(frame)
            if idx % 200 == 0:
                elapsed = time.time() - t_start
                fps = (idx + 1) / elapsed if elapsed > 0 else 0
                pos = pipeline.trajectory[-1]
                print(f'  [{idx:4d}/{n_frames}]  fps={fps:.1f}  pos=[{pos[0]:.1f}, {pos[1]:.1f}, {pos[2]:.1f}]')

        elapsed = time.time() - t_start
        print(f'  Finished {n_frames} frames in {elapsed:.1f}s  ({n_frames/elapsed:.1f} fps)')

        traj_est = pipeline.get_trajectory()
        traj_gt  = seq.get_gt_xyz()
        metrics  = {'traj_est': traj_est, 'traj_gt': traj_gt}

        if traj_gt is not None:
            n = min(len(traj_est), len(traj_gt))
            ate   = compute_ate(traj_est[:n], traj_gt[:n])
            rpe   = compute_rpe(traj_est[:n], traj_gt[:n], delta=1)
            drift = compute_drift(traj_est[:n], traj_gt[:n])
            metrics.update({'ate': ate, 'rpe': rpe, 'drift': drift, 'n': n})
            print_metrics(ate, rpe, drift, label=f'Sequence {seq_id}')
        
        all_metrics[seq_id] = metrics

    except Exception as e:
        import traceback
        print(f'  [ERROR] seq {seq_id}: {e}')
        traceback.print_exc()

print('\nAll sequences done.')


  KITTI Sequence 00
  Using camera_id=2
  [   0/4541]  fps=14.7  pos=[0.0, 0.0, 0.0]


[INFO] Frame    1 | kps= 590 | matches= 337 | inliers= 186 | pos=[-0.02 -0.01 -1.00]
[INFO] Frame    2 | kps= 618 | matches= 333 | inliers= 208 | pos=[-0.00 -0.02 -2.00]
[INFO] Frame    3 | kps= 587 | matches= 333 | inliers= 220 | pos=[0.01 -0.04 -3.00]
[INFO] Frame    4 | kps= 612 | matches= 346 | inliers= 212 | pos=[0.02 -0.00 -4.00]
[INFO] Frame    5 | kps= 597 | matches= 365 | inliers= 236 | pos=[-0.00 0.05 -5.00]
[INFO] Frame    6 | kps= 601 | matches= 348 | inliers= 183 | pos=[0.03 0.06 -6.00]
[INFO] Frame    7 | kps= 590 | matches= 342 | inliers= 231 | pos=[0.08 0.05 -7.00]
[INFO] Frame    8 | kps= 579 | matches= 328 | inliers= 226 | pos=[0.09 0.05 -7.99]
[INFO] Frame    9 | kps= 514 | matches= 311 | inliers= 216 | pos=[-0.05 0.02 -8.99]
[INFO] Frame   10 | kps= 517 | matches= 285 | inliers= 181 | pos=[-0.08 0.07 -9.98]
[INFO] Frame   11 | kps= 497 | matches= 324 | inliers= 216 | pos=[-0.13 0.08 -10.98]
[INFO] Frame   12 | kps= 463 | matches= 315 | inliers= 250 | pos=[-0.13 0.04

  [ 200/4541]  fps=15.7  pos=[84.9, 4.1, -115.7]


[INFO] Frame  204 | kps= 706 | matches= 240 | inliers= 188 | pos=[88.71 4.22 -116.73]
[INFO] Frame  205 | kps= 744 | matches= 260 | inliers= 146 | pos=[89.63 4.29 -117.12]
[INFO] Frame  206 | kps= 672 | matches= 305 | inliers= 131 | pos=[90.51 4.31 -117.60]
[INFO] Frame  207 | kps= 654 | matches= 327 | inliers= 166 | pos=[91.33 4.34 -118.16]
[INFO] Frame  208 | kps= 624 | matches= 349 | inliers= 152 | pos=[92.10 4.39 -118.81]
[INFO] Frame  209 | kps= 649 | matches= 340 | inliers= 180 | pos=[92.86 4.39 -119.45]
[INFO] Frame  210 | kps= 585 | matches= 325 | inliers= 154 | pos=[93.61 4.41 -120.11]
[INFO] Frame  211 | kps= 500 | matches= 313 | inliers= 163 | pos=[94.31 4.45 -120.82]
[INFO] Frame  212 | kps= 519 | matches= 308 | inliers= 151 | pos=[94.96 4.44 -121.59]
[INFO] Frame  213 | kps= 481 | matches= 300 | inliers= 114 | pos=[95.53 4.52 -122.40]
[INFO] Frame  214 | kps= 474 | matches= 321 | inliers= 151 | pos=[96.03 4.60 -123.27]
[INFO] Frame  215 | kps= 452 | matches= 319 | inliers=

  [ 400/4541]  fps=16.6  pos=[117.0, 13.6, -306.0]


...

[INFO] Frame 2203 | kps= 947 | matches= 368 | inliers= 305 | pos=[75.79 22.50 -484.73]
[INFO] Frame 2204 | kps= 915 | matches= 354 | inliers= 275 | pos=[75.89 22.55 -483.73]
[INFO] Frame 2205 | kps= 838 | matches= 353 | inliers= 288 | pos=[75.93 22.50 -482.73]
[INFO] Frame 2206 | kps= 948 | matches= 340 | inliers= 242 | pos=[76.00 22.50 -481.74]
[INFO] Frame 2207 | kps=1059 | matches= 364 | inliers= 311 | pos=[76.07 22.51 -480.74]
[INFO] Frame 2208 | kps=1080 | matches= 361 | inliers= 295 | pos=[76.13 22.50 -479.74]
[INFO] Frame 2209 | kps= 926 | matches= 374 | inliers= 300 | pos=[76.16 22.48 -478.74]
[INFO] Frame 2210 | kps= 959 | matches= 353 | inliers= 273 | pos=[76.22 22.50 -477.74]
[INFO] Frame 2211 | kps=1064 | matches= 327 | inliers= 265 | pos=[76.24 22.50 -476.74]
[INFO] Frame 2212 | kps=1163 | matches= 306 | inliers= 252 | pos=[76.29 22.51 -475.74]
[INFO] Frame 2213 | kps=1187 | matches= 307 | inliers= 233 | pos=[76.32 22.53 -474.74]
[INFO] Frame 2214 | kps=1239 | matches= 310

  [2400/4661]  fps=14.2  pos=[93.9, 26.4, -289.0]


[INFO] Frame 2404 | kps= 564 | matches= 240 | inliers= 164 | pos=[94.90 26.66 -285.16]
[INFO] Frame 2405 | kps= 477 | matches= 238 | inliers= 195 | pos=[95.17 26.71 -284.20]
[INFO] Frame 2406 | kps= 472 | matches= 219 | inliers= 155 | pos=[95.44 26.77 -283.24]
[INFO] Frame 2407 | kps= 483 | matches= 239 | inliers= 194 | pos=[95.74 26.79 -282.28]
[INFO] Frame 2408 | kps= 484 | matches= 242 | inliers= 191 | pos=[96.03 26.83 -281.33]
[INFO] Frame 2409 | kps= 495 | matches= 228 | inliers= 173 | pos=[96.33 26.87 -280.37]
[INFO] Frame 2410 | kps= 465 | matches= 231 | inliers= 157 | pos=[96.69 26.94 -279.44]
[INFO] Frame 2411 | kps= 495 | matches= 201 | inliers= 176 | pos=[97.04 26.97 -278.51]
[INFO] Frame 2412 | kps= 466 | matches= 213 | inliers= 168 | pos=[97.38 27.02 -277.57]
[INFO] Frame 2413 | kps= 489 | matches= 179 | inliers= 120 | pos=[97.73 27.00 -276.63]
[INFO] Frame 2414 | kps= 481 | matches= 201 | inliers= 138 | pos=[98.11 27.08 -275.71]
[INFO] Frame 2415 | kps= 465 | matches= 218

  [2600/4661]  fps=14.3  pos=[206.3, 22.8, -181.6]


[INFO] Frame 2604 | kps= 486 | matches= 180 | inliers= 144 | pos=[203.77 22.55 -178.48]
[INFO] Frame 2605 | kps= 471 | matches= 170 | inliers= 120 | pos=[203.19 22.51 -177.67]
[INFO] Frame 2606 | kps= 516 | matches= 183 | inliers= 130 | pos=[202.58 22.46 -176.87]
[INFO] Frame 2607 | kps= 544 | matches= 200 | inliers= 151 | pos=[202.00 22.42 -176.06]
[INFO] Frame 2608 | kps= 475 | matches= 212 | inliers= 181 | pos=[201.42 22.36 -175.25]
[INFO] Frame 2609 | kps= 500 | matches= 191 | inliers= 142 | pos=[200.83 22.32 -174.44]
[INFO] Frame 2610 | kps= 364 | matches= 186 | inliers= 155 | pos=[200.25 22.27 -173.63]
[INFO] Frame 2611 | kps= 495 | matches= 172 | inliers= 126 | pos=[199.68 22.23 -172.81]
[INFO] Frame 2612 | kps= 507 | matches= 226 | inliers= 158 | pos=[199.10 22.14 -172.00]
[INFO] Frame 2613 | kps= 462 | matches= 226 | inliers= 181 | pos=[198.52 22.11 -171.18]
[INFO] Frame 2614 | kps= 489 | matches= 200 | inliers= 149 | pos=[197.97 22.03 -170.35]
[INFO] Frame 2615 | kps= 571 | m

## Cell 8 — Summary metrics table

In [39]:
import pandas as pd

rows = []
for seq_id, m in all_metrics.items():
    if 'ate' not in m:
        continue
    rows.append({
        'Seq':               seq_id,
        'ATE RMSE (m)':      round(m['ate']['rmse'], 3),
        'ATE mean (m)':      round(m['ate']['mean'], 3),
        'RPE trans RMSE (m)':round(m['rpe']['trans_rmse'], 4),
        'RPE rot RMSE (deg)':round(m['rpe']['rot_rmse'], 3),
        'Drift %':           round(m['drift']['translational_drift_pct'], 2),
        'GT dist (m)':       round(m['drift']['total_distance_m'], 1),
    })

if rows:
    df = pd.DataFrame(rows).set_index('Seq')
    display(df)
    df.to_csv(f'{OUTPUT_DIR}/summary.csv')
    print(f'Saved to {OUTPUT_DIR}/summary.csv')
else:
    print('No metrics — no ground-truth poses were found.')

No metrics — no ground-truth poses were found.


## Cell 9 — Trajectory plots (all sequences)

In [40]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

seq_ids = [s for s in all_metrics
           if (Path(OUTPUT_DIR) / s / 'trajectory.png').exists()]

if not seq_ids:
    print('No trajectory plots found.')
else:
    cols = min(3, len(seq_ids))
    rows = (len(seq_ids) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
    axes_flat = [axes] if len(seq_ids) == 1 else axes.flatten()

    for ax, sid in zip(axes_flat, seq_ids):
        img = mpimg.imread(f'{OUTPUT_DIR}/{sid}/trajectory.png')
        ax.imshow(img); ax.set_title(f'Seq {sid}', fontsize=11); ax.axis('off')

    for ax in axes_flat[len(seq_ids):]:
        ax.set_visible(False)

    plt.suptitle('Estimated vs ground-truth trajectories', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/all_trajectories.png', dpi=120, bbox_inches='tight')
    plt.show()

No trajectory plots found.


## Cell 10 — List all output files

In [41]:
import os
from pathlib import Path

print(f'Output files in {OUTPUT_DIR}:\n')
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs.sort()
    level  = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        size_kb = Path(root, f).stat().st_size // 1024
        print(f'{indent}  {f}  ({size_kb} KB)')

print('\nTo download results:')
print('  Option A: File browser (left sidebar) → /kaggle/working/results → right-click → Download')
print('  Option B: Click Save Version → Output tab → download as zip')

Output files in /kaggle/working/results:

results/

To download results:
  Option A: File browser (left sidebar) → /kaggle/working/results → right-click → Download
  Option B: Click Save Version → Output tab → download as zip
